<a href="https://colab.research.google.com/github/HashamHassan-01/flyrank-ml-internship-hasham/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HashamHassan-01/flyrank-ml-internship-hasham/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*
## ML Task Type

Task Type: Scoring (Regression)

The goal is to assign each content page an opportunity score that helps prioritize which pages should be reviewed first for refresh. Pages with higher predicted opportunity should be reviewed before lower-priority pages.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("flyrank-ml-internship-hasham/data/raw/content_refresh_anonymized.csv")
print("Dataset Shape:", df.shape)
print("\nFirst 10 columns:")
print(df.columns[:10])

Dataset Shape: (30000, 44)

First 10 columns:
Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count'],
      dtype='object')


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
## Target or Proxy

The target is a content opportunity score that estimates which pages are most likely to benefit from a content refresh.

There is no observed "refresh success" label in the current dataset — for example,
there's no record of which pages were refreshed and whether that refresh actually
improved performance afterward. To make progress on framing this problem now, this
notebook defines a proxy opportunity_score using percentile ranks of search_volume,
ctr, engagement_rate, trend_pct, and content_age_days.

This is a rule-based construction used for framing purposes, not a substitute for a
true observed outcome. A production version of this system would prefer an observed
target instead — for example, tracking the change in sessions/CTR 90 days after a
page is refreshed, and using that measured improvement as the real label for training.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Use percentile rank (0 to 1) instead of raw values so different scales don't distort the formula
# e.g. search_volume can be 0-10000+, ctr is 0-1 — ranks make them comparable

demand_signal      = df["search_volume"].rank(pct=True)
underperformance   = 1 - df["ctr"].rank(pct=True)          # low ctr = high underperformance
low_engagement     = 1 - df["engagement_rate"].rank(pct=True)
decline_signal     = 1 - df["trend_pct"].rank(pct=True)    # negative trend = higher rank here
age_signal         = df["content_age_days"].rank(pct=True)

df["opportunity_score"] = (
    demand_signal    * 0.35 +
    underperformance * 0.25 +
    low_engagement   * 0.20 +
    decline_signal   * 0.10 +
    age_signal       * 0.10
)

df["opportunity_score"].describe()

print("TOP 5 opportunity pages:")
print(df.nlargest(5, "opportunity_score")[
    ["content_id","search_volume","ctr","engagement_rate","trend_pct","content_age_days"]
])

print("\nBOTTOM 5 opportunity pages:")
print(df.nsmallest(5, "opportunity_score")[
    ["content_id","search_volume","ctr","engagement_rate","trend_pct","content_age_days"]
])

TOP 5 opportunity pages:
                 content_id  search_volume  ctr  engagement_rate  trend_pct  \
7500   content_5b5e85993c2b         1300.0  0.0              0.0     -100.0   
28449  content_e8fc703f7ef0         9900.0  0.0              0.0      -96.5   
1611   content_a37cded67cd7         2900.0  0.0              0.0     -100.0   
25608  content_b933ddb19b17         3600.0  0.0              0.0      -91.5   
2288   content_c3ccadb77bc4         1600.0  0.0              0.0      -93.2   

       content_age_days  
7500                557  
28449               463  
1611                460  
25608               466  
2288                482  

BOTTOM 5 opportunity pages:
                 content_id  search_volume    ctr  engagement_rate  trend_pct  \
10796  content_0be88b7c1f24            0.0  26.47            62.50       87.5   
17800  content_b668017bde81            0.0   1.98             6.49      143.8   
20128  content_91ebaad39a3f            0.0   3.68            20.00      

## 3. Success metric

*One metric you can defend. What number means 'good'?*

## Success Metric


Because the business goal is about getting the top-priority pages right — not just
producing a number close to some target — the success metric for this task is
Precision@20: of the model's top 20 predicted opportunity pages, how many are
actually in the true top 20. This directly measures whether the content team's
refresh queue would be built from the correct pages.

(MAE could be used as a training-time loss function while fitting a regression
model, but it isn't the metric being defended here — ranking correctness is what
matters for this use case.)

**Action this output supports:** The opportunity_score ranks all content pages, and
the content team pulls the top N pages from this ranked list as their monthly/
quarterly refresh queue — replacing manual, unordered review.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from scipy.stats import spearmanr

# Later, once you have predictions from a real model:
# corr, p_value = spearmanr(y_true, y_pred)

# Quick sanity check: does the function work correctly?
# Using opportunity_score against itself should give perfect overlap (1.0)
test_score = precision_at_k(df["opportunity_score"], df["opportunity_score"], k=20)
print("Precision@20 (self-check, should be 1.0):", test_score)


Precision@20 (self-check, should be 1.0): 1.0


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## Unit of Analysis

One row in the dataset represents one content page (article). Each row contains information about that page, including search performance, content characteristics, engagement metrics, and historical traffic. The ML model will evaluate each content page individually to determine its priority for refresh.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,opportunity_score
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,0.330401
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,0.725130
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,0.419052
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0.397639
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,0.407672


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple fixed rule cannot reliably identify which pages should be refreshed because many factors influence content performance. A page may have low traffic but high engagement, or high search volume but strong competition. Machine learning can evaluate multiple signals such as CTR, engagement rate, search volume, content age, historical traffic, and competition together to identify patterns that are difficult to capture with a single if-statement. This allows content teams to prioritize pages more accurately for refresh.
This is confirmed by the correlation check below — no single feature correlates
above 0.40 with opportunity_score, so a simple threshold rule on any one column
would miss most high-opportunity pages.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
corr_matrix = df[[
    "search_volume","competition","ctr","engagement_rate",
    "content_age_days","trend_pct","opportunity_score"
]].corr()

print(corr_matrix["opportunity_score"].sort_values(ascending=False))

opportunity_score    1.000000
content_age_days     0.396707
competition          0.303348
search_volume        0.146678
trend_pct           -0.024431
ctr                 -0.187568
engagement_rate     -0.239628
Name: opportunity_score, dtype: float64


## Self-check

✅ Every section is filled with markdown and supporting code.

✅ The notebook runs from top to bottom with no errors.

✅ No client names, URLs, or private queries are included.

✅ Your wording is careful (observed, measured, decision-support).

✅ The notebook has been committed under work/notebooks/.